In [ ]:
%pip install openai

In [ ]:
import json
import os
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = ROOT / "data" / "processed"

api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key) 

MODEL = "gpt-4o-mini"
print("Client ready")

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train = load_jsonl(PROCESSED_DIR / "python_train_filtered.jsonl")
print(f"Loaded {len(train):,} filtered Python train samples")


s = train[0]
print("\nPatch (first 200):", s['patch'][:200])
print("\nComment:", s['msg'])

In [ ]:
def build_prompt(patch, msg, oldf_snippet=""):
    """CoT reasoning generate karne ka prompt."""
    return f"""You are an experienced code reviewer. You are given a code change \
and the final review comment a reviewer wrote. Reconstruct the step-by-step \
reasoning that leads to this comment.

Write 2 to 4 short reasoning steps. The reasoning must logically conclude in \
the exact comment provided. Keep it concise and technical.

Code change:
{patch[:1500]}

Final reviewer comment:
{msg}

Respond in exactly this format:
<reasoning>
your 2-4 step reasoning here
</reasoning>
<comment>
{msg}
</comment>"""


def generate_cot(patch, msg):
    prompt = build_prompt(patch, msg)
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,    
        max_tokens=300,
    )
    return resp.choices[0].message.content

In [ ]:

print("Testing on 5 samples...\n")

for i in range(5):
    s = train[i]
    output = generate_cot(s['patch'], s['msg'])
    print("=" * 70)
    print(f"SAMPLE {i+1}")
    print(f"Actual comment: {s['msg']}")
    print("-" * 70)
    print(output)
    print()

In [ ]:
import json
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()                              
api_key = os.getenv("OPENAI_API_KEY")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = ROOT / "data" / "processed"
BATCH_DIR = ROOT / "data" / "batch"
BATCH_DIR.mkdir(parents=True, exist_ok=True)

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"
print("Ready")

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train = load_jsonl(PROCESSED_DIR / "python_train_filtered.jsonl")
print(f"Loaded {len(train):,} samples")

def build_prompt(patch, msg):
    return f"""You are an experienced code reviewer. You are given a code change \
and the final review comment a reviewer wrote. Reconstruct the step-by-step \
reasoning that leads to this comment.

Write 2 to 4 short reasoning steps. The reasoning must logically conclude in \
the exact comment provided. Keep it concise and technical.

Code change:
{patch[:1500]}

Final reviewer comment:
{msg}

Respond in exactly this format:
<reasoning>
your 2-4 step reasoning here
</reasoning>
<comment>
{msg}
</comment>"""

In [ ]:
batch_file = BATCH_DIR / "cot_requests.jsonl"

with open(batch_file, "w", encoding="utf-8") as f:
    for i, s in enumerate(train):
        request = {
            "custom_id": f"sample-{i}",          
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": MODEL,
                "messages": [
                    {"role": "user", "content": build_prompt(s['patch'], s['msg'])}
                ],
                "temperature": 0.3,
                "max_tokens": 300,
            }
        }
        f.write(json.dumps(request) + "\n")

print(f"Created batch file with {len(train):,} requests")
print(f"File: {batch_file}")

In [ ]:

with open(batch_file, "rb") as f:
    uploaded = client.files.create(file=f, purpose="batch")
print(f"Uploaded file id: {uploaded.id}")


batch_job = client.batches.create(
    input_file_id=uploaded.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
)
print(f"\nBatch job started!")
print(f"Job ID: {batch_job.id}")
print(f"Status: {batch_job.status}")


print(f"\n>>> SAVE THIS JOB ID: {batch_job.id} <<<")


with open(BATCH_DIR / "job_id.txt", "w") as f:
    f.write(batch_job.id)
print("Job ID saved to job_id.txt")

In [ ]:
job = client.batches.retrieve(job_id)

print("Status:", job.status)
print("\n--- Error details ---")
print("Errors:", job.errors)

# Poora object dekho (saari fields)
print("\n--- Full object ---")
for key, value in vars(job).items():
    print(f"{key}: {value}")

In [ ]:
pip install python-dotenv

In [ ]:
import json
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BATCH_DIR = ROOT / "data" / "batch"
PROCESSED_DIR = ROOT / "data" / "processed"
BATCH_DIR.mkdir(parents=True, exist_ok=True)
api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"

def load_jsonl(path):
    return [json.loads(line) for line in open(path, encoding="utf-8")]

train = load_jsonl(PROCESSED_DIR / "python_train_filtered.jsonl")

def build_prompt(patch, msg):
    return f"""You are an experienced code reviewer. You are given a code change \
and the final review comment a reviewer wrote. Reconstruct the step-by-step \
reasoning that leads to this comment.

Write 2 to 4 short reasoning steps. The reasoning must logically conclude in \
the exact comment provided. Keep it concise and technical.

Code change:
{patch[:1200]}

Final reviewer comment:
{msg}

Respond in exactly this format:
<reasoning>
your 2-4 step reasoning here
</reasoning>
<comment>
{msg}
</comment>"""

CHUNK_SIZE = 3000
chunks = [train[i:i+CHUNK_SIZE] for i in range(0, len(train), CHUNK_SIZE)]
print(f"Total: {len(train):,} | Chunks: {len(chunks)}")

In [ ]:
def submit_chunk(chunk_idx):
    chunk = chunks[chunk_idx]
    req_file = BATCH_DIR / f"requests_chunk{chunk_idx}.jsonl"

    with open(req_file, "w", encoding="utf-8") as f:
        for i, s in enumerate(chunk):
            global_idx = chunk_idx * CHUNK_SIZE + i   # train[global_idx] = ye sample
            req = {
                "custom_id": f"row-{global_idx}",      # unique, aur seedha train index
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": MODEL,
                    "messages": [{"role": "user",
                                  "content": build_prompt(s['patch'], s['msg'])}],
                    "temperature": 0.3,
                    "max_tokens": 300,
                }
            }
            f.write(json.dumps(req) + "\n")

    with open(req_file, "rb") as f:
        uploaded = client.files.create(file=f, purpose="batch")
    batch = client.batches.create(
        input_file_id=uploaded.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )
    with open(BATCH_DIR / f"batch_chunk{chunk_idx}.txt", "w") as f:
        f.write(batch.id)
    print(f"Chunk {chunk_idx} submitted: {batch.id}")
    return batch.id

In [ ]:
submit_chunk(0)

In [ ]:
bid = open(BATCH_DIR / "batch_chunk0.txt").read().strip()
job = client.batches.retrieve(bid)
print(f"Status: {job.status} | {job.request_counts.completed}/{job.request_counts.total}")
if job.status == "failed":
    print(job.errors)

In [ ]:
submit_chunk(1)

In [ ]:
# Chunk 0 ka result download
bid = open(BATCH_DIR / "batch_chunk0.txt").read().strip()
job = client.batches.retrieve(bid)

result = client.files.content(job.output_file_id)
out_path = BATCH_DIR / "results_chunk0.jsonl"
with open(out_path, "wb") as f:
    f.write(result.content)

results = [json.loads(line) for line in open(out_path, encoding="utf-8")]
print(f"Downloaded {len(results):,} results")

In [ ]:
for r in results[:3]:
    idx = int(r["custom_id"].split("-")[1])  
    original = train[idx]
    output = r["response"]["body"]["choices"][0]["message"]["content"]
    
    print("=" * 70)
    print(f"custom_id: {r['custom_id']} -> train[{idx}]")
    print(f"Original comment: {original['msg']}")
    print("-" * 70)
    print(output)
    print()

In [ ]:
submit_chunk(2)

In [ ]:
bid = open(BATCH_DIR / "batch_chunk7.txt").read().strip()
job = client.batches.retrieve(bid)
print(f"Status: {job.status} | {job.request_counts.completed}/{job.request_counts.total}")

In [ ]:
submit_chunk(5)

In [ ]:
submit_chunk(7)

In [ ]:
for idx in range(len(chunks)):
    f = BATCH_DIR / f"batch_chunk{idx}.txt"
    if not f.exists():
        print(f"Chunk {idx}: not submitted yet")
        continue
    bid = open(f).read().strip()
    job = client.batches.retrieve(bid)
    line = f"Chunk {idx}: {job.status} | {job.request_counts.completed}/{job.request_counts.total}"
    if job.status == "failed":
        line += f"  ERROR: {job.errors.data[0].code if job.errors and job.errors.data else '?'}"
    print(line)

In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BATCH_DIR = ROOT / "data" / "batch"
PROCESSED_DIR = ROOT / "data" / "processed"

# train load (agar setup cell se already nahi)
def load_jsonl(path):
    return [json.loads(line) for line in open(path, encoding="utf-8")]
train = load_jsonl(PROCESSED_DIR / "python_train_filtered.jsonl")

# Helper: nested result se actual text nikalo
def get_text(result):
    return result["response"]["body"]["choices"][0]["message"]["content"]

# Dictionary: global_index -> reasoning ka raw output
cot_by_index = {}

for idx in range(8):
    # 1. is chunk ka batch ID file se padho
    bid = open(BATCH_DIR / f"batch_chunk{idx}.txt").read().strip()
    # 2. job se output file ka pata lo
    job = client.batches.retrieve(bid)
    # 3. result download karo
    result = client.files.content(job.output_file_id)
    # 4. local file mein save karo (backup, taaki dobara download na karna pade)
    out_path = BATCH_DIR / f"results_chunk{idx}.jsonl"
    with open(out_path, "wb") as f:
        f.write(result.content)
    # 5. har line parse karo, reasoning nikaalo, index se store karo
    for line in open(out_path, encoding="utf-8"):
        r = json.loads(line)
        gidx = int(r["custom_id"].split("-")[1])   # "row-5" -> 5
        cot_by_index[gidx] = get_text(r)
    print(f"Chunk {idx} downloaded + parsed")

print(f"\nTotal CoT outputs: {len(cot_by_index):,}")

In [ ]:
import re

def extract_parts(text):
    """Output se reasoning aur comment alag nikalo (regex se)."""
    r_match = re.search(r"<reasoning>(.*?)</reasoning>", text, re.DOTALL)
    c_match = re.search(r"<comment>(.*?)</comment>", text, re.DOTALL)
    reasoning = r_match.group(1).strip() if r_match else None
    comment   = c_match.group(1).strip() if c_match else None
    return reasoning, comment

valid_samples = []
rejected = {"no_reasoning": 0, "no_comment": 0, "too_short": 0, "comment_mismatch": 0}

for i, s in enumerate(train):
    if i not in cot_by_index:
        continue
    reasoning, comment = extract_parts(cot_by_index[i])

    # Check 1: reasoning hai?
    if not reasoning:
        rejected["no_reasoning"] += 1
        continue
    # Check 2: comment hai?
    if not comment:
        rejected["no_comment"] += 1
        continue
    # Check 3: reasoning kaafi lambi?
    if len(reasoning.split()) < 10:
        rejected["too_short"] += 1
        continue
    # Check 4: comment original se match karta hai?
    if comment[:20].lower() not in s["msg"].lower() and s["msg"][:20].lower() not in comment.lower():
        rejected["comment_mismatch"] += 1
        continue

    # Sab pass — sample mein reasoning add karo
    new_sample = dict(s)              # original sample copy karo
    new_sample["reasoning"] = reasoning   # naya field add
    valid_samples.append(new_sample)

print(f"Valid CoT samples : {len(valid_samples):,}")
print(f"Rejected          : {sum(rejected.values()):,}")
for reason, count in rejected.items():
    print(f"  {reason:18s}: {count:,}")

In [ ]:
import random
random.seed(42)

# 5 random valid samples dekho (baaki scroll kar sakta)  
check = random.sample(valid_samples, 5)

for i, s in enumerate(check):
    print("=" * 70)
    print(f"SAMPLE {i+1}")
    print(f"Original comment: {s['msg'][:120]}")
    print("-" * 70)
    print(f"Reasoning:\n{s['reasoning']}")
    print()

In [ ]:
def save_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    print(f"Saved {len(rows):,} -> {path}")

save_jsonl(valid_samples, PROCESSED_DIR / "python_train_cot.jsonl")

In [ ]:
import json
s = valid_samples[0]
print("Fields:", list(s.keys()))
print("\nHas reasoning?", "reasoning" in s)
print("Reasoning length:", len(s["reasoning"].split()), "words")